In [1]:
import os, sys
os.environ['R_HOME'] = sys.exec_prefix + '/lib/R/'
import scanpy as sc
import scFates as scf

In [ ]:
geo = 'GSE175435'
pth_data = os.path.join('..', '..', 'data')
pth_unzip = os.path.join(pth_data, 'unzip', geo)
pth_out = os.path.join(pth_data, 'cancer', geo)

In [ ]:
adata = sc.read_loom(os.path.join(pth_unzip, 'GSM5333520_LglIR.loom'))
adata

In [ ]:
adata.var['mt'] = adata.var_names.str.startswith('mt:')
sc.pp.calculate_qc_metrics(adata, qc_vars = ['mt'],
                           percent_top = None,
                           inplace = True,
                           log1p = False)
sc.pl.violin(adata, ['n_genes_by_counts',
                     'total_counts',
                     'pct_counts_mt'],
             jitter = .4, multi_panel = True)

In [ ]:
sc.pp.filter_cells(adata, min_genes = 600)
sc.pp.filter_cells(adata, max_genes = 1800)
sc.pp.filter_cells(adata, max_counts = 15000)
adata = adata[adata.obs.pct_counts_mt < 4.5].copy()
adata

In [ ]:
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)
sc.pp.filter_genes(adata, min_cells = 3)
scf.pp.find_overdispersed(adata, plot = True)

In [ ]:
sc.pp.pca(adata)
sc.pp.neighbors(adata, n_neighbors = 50)
sc.tl.leiden(adata)
sc.tl.umap(adata)
sc.pl.umap(adata, color = 'leiden', size = 30,
    legend_loc = 'on data', legend_fontoutline = True)

In [ ]:
genes = ['osk', 'vas', 'Zasp66', 'Hml']
sc.pl.umap(adata, color = genes, size = 50)

In [ ]:
remove = ['7', '8', '9', '10']
adata = adata[~adata.obs.leiden.isin(remove)].copy()
adata

In [ ]:
sc.tl.leiden(adata, resolution = 2.5)
sc.pl.umap(adata, color = 'leiden', size = 30,
    legend_loc = 'on data', legend_fontoutline = True)

In [ ]:
sc.pl.dotplot(adata, 'hh', 'leiden', swap_axes = True)

In [ ]:
adata = adata[adata.obs.leiden != '17'].copy()
adata

In [ ]:
sc.tl.leiden(adata)
sc.pl.umap(adata, color = 'leiden', size = 30,
    legend_loc = 'on data', legend_fontoutline = True)

In [ ]:
sc.pl.dotplot(adata,
              {'Polar'     : ['upd1'],
               'Stretched' : ['dpp'],
            #    'Border'    : ['slbo'],
               'Mitotic'   : ['ct']},
              'leiden')

In [ ]:
celltype_dict = {
    '17' : 'nan',
    '13' : 'Polar',
    '15' : 'Stretched',
    '16' : 'Stretched',
    '22' : 'Border',
    '0'  : 'Mitotic',
    '1'  : 'Mitotic',
    '2'  : 'Other',
    '3'  : 'Other',
    '4'  : 'Other',
    '5'  : 'Other',
    '6'  : 'Other',
    '7'  : 'Other',
    '8'  : 'Other',
    '9'  : 'Other',
    '10' : 'Other',
    '11' : 'Other',
    '12' : 'Other',
    '14' : 'Other',
    '18' : 'Other',
    '19' : 'Other',
    '20' : 'Other',
    '21' : 'Other'}

adata.obs['celltype'] = adata.obs.leiden.map(celltype_dict).astype('category')

In [ ]:
sc.pl.umap(adata, color = 'celltype', size = 30)

In [ ]:
import glob
import pandas as pd
pth_feat = os.path.join('..', 'data', 'features', 'biomart')

feat_fn = sorted(glob.glob(os.path.join(pth_feat, '*.csv')))
get_src = lambda fn: os.path.split(fn)[1].replace('.csv', '')
feat_dict = {get_src(fn) : pd.read_csv(fn) for fn in feat_fn}

# module signatures
g_ref = feat_dict['union'].dmelanogaster.dropna().unique()
for src, df_src in feat_dict.items():
    if src != 'union':
        print(src)
        g = df_src.dmelanogaster.dropna().unique()
        sc.tl.score_genes(adata,
                            gene_list = g,
                            score_name = src,
                            ctrl_as_ref = False,
                            gene_pool = g_ref,
                            random_state = 1234)

In [ ]:
fn = os.path.join(pth_data, 'summary.csv')
df_summary = pd.read_csv(fn, index_col = 0)
labels_dict = (df_summary.Signature.dropna() + '_mod').to_dict()
labels_dict

In [ ]:
import matplotlib.pyplot as plt

src = list(labels_dict.keys())
mod = list(labels_dict.values())
adata.obs[mod] = adata.obs[src].abs()

# signatures dotplot
fs = 7
dp = sc.pl.dotplot(adata, mod, 'celltype',
                #    categories_order = ix,
                   expression_cutoff = .1,
                   cmap = 'inferno',
                   figsize = (2.5, .7),
                   swap_axes = True,
                   return_fig = True,
                   show = False)

dp.style(largest_dot = 110,
         x_padding = .6,
         y_padding = .6)
dp.legend(width = .8)

ax = dp.get_axes()
ax_main = ax['mainplot_ax']
ax_size = ax['size_legend_ax']
cax = ax['color_legend_ax']

ax_size.set_title('% Cells', size = fs, pad = -8)
ax_size.tick_params('x', size = 0, labelsize = fs, pad = 2)
cax.set_title('Signature\nMagnitude', size = fs - .25, pad = 4.75)
# cax.set_xticks([.05, .15])
cax.tick_params('x', size = 0, labelsize = fs)

pos_main = ax_main.get_position()
pos_size = ax_size.get_position()
pos_color = cax.get_position()

ax_size.set_position([.985 * pos_size.x0,
                      pos_main.y0 + .465 * pos_main.height,
                      pos_size.width,
                      .6 * pos_main.height])

cax.set_position([.985 * pos_color.x0,
                  pos_main.y0 - .48 * pos_main.height,
                  pos_color.width,
                  .15 * pos_main.height])

ax_main.set_yticklabels(ax_main.get_yticklabels(),
                        fontsize = fs + .5)

ax_main.set_xticklabels(ax_main.get_xticklabels(),
                        rotation = -55, fontsize = fs,
                        rotation_mode = 'anchor',
                        ha = 'left')

# fn = os.path.join(pth_out, 'signatures.pdf')
# plt.savefig(fn, bbox_inches = 'tight')
plt.show()